# Customer Churn Analytics Platform

## 04 — Feature Engineering

**Objective:**
Create analytical and business-oriented features from the cleaned customer churn dataset to support deeper churn, revenue, and customer risk analysis.

### Feature Engineering Goals

* Create binary churn and active customer flags
* Segment customers by tenure
* Categorize monthly and total charges
* Create service adoption indicators
* Calculate total service count
* Create customer revenue metrics
* Calculate revenue per tenure
* Identify monthly revenue at risk
* Build a customer risk score
* Classify customers into risk categories
* Validate all engineered features
* Save the feature-engineered dataset for downstream analysis

In [1]:
# ============================================================
# CELL 01 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# ============================================================
# LOAD CLEAN DATASET
# ============================================================

DATA_PATH = Path(
    "../data/processed/customer_churn_clean.csv"
)

df = pd.read_csv(DATA_PATH)

print("Clean dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

Clean dataset loaded successfully.
Rows    : 7,043
Columns : 21


In [4]:
# ============================================================
# INITIAL DATA VALIDATION
# ============================================================

print("=" * 60)
print("INITIAL DATA VALIDATION")
print("=" * 60)

print(f"Rows             : {len(df):,}")
print(f"Columns          : {len(df.columns):,}")
print(f"Missing Values   : {df.isna().sum().sum():,}")
print(f"Duplicate Rows   : {df.duplicated().sum():,}")
print(f"Unique Customers : {df['customerid'].nunique():,}")

INITIAL DATA VALIDATION
Rows             : 7,043
Columns          : 21
Missing Values   : 0
Duplicate Rows   : 0
Unique Customers : 7,043


In [5]:
# ============================================================
# CREATE WORKING COPY
# ============================================================

df_features = df.copy()

print("Feature engineering working copy created.")

Feature engineering working copy created.


In [6]:
# ============================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================

required_columns = [
    "customerid",
    "seniorcitizen",
    "tenure",
    "phoneservice",
    "onlinesecurity",
    "onlinebackup",
    "deviceprotection",
    "techsupport",
    "streamingtv",
    "streamingmovies",
    "contract",
    "paymentmethod",
    "internetservice",
    "monthlycharges",
    "totalcharges",
    "churn"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_features.columns
]

if len(missing_columns) == 0:
    print("All required columns are available.")
else:
    print("Missing columns:")
    print(missing_columns)

All required columns are available.


# CUSTOMER STATUS FEATURES

In [7]:
# ============================================================
# CREATE CHURN FLAG
# ============================================================

df_features["churn_flag"] = (
    df_features["churn"]
    .astype(str)
    .str.lower()
    .eq("yes")
    .astype(int)
)

print("Churn flag created.")

print("\nChurn Flag Distribution:")
print(
    df_features["churn_flag"]
    .value_counts()
    .sort_index()
)

Churn flag created.

Churn Flag Distribution:
churn_flag
0    5174
1    1869
Name: count, dtype: int64


In [8]:
# ============================================================
# CREATE ACTIVE FLAG
# ============================================================

df_features["active_flag"] = (
    df_features["churn"]
    .astype(str)
    .str.lower()
    .eq("no")
    .astype(int)
)

print("Active flag created.")

print("\nActive Flag Distribution:")
print(
    df_features["active_flag"]
    .value_counts()
    .sort_index()
)

Active flag created.

Active Flag Distribution:
active_flag
0    1869
1    5174
Name: count, dtype: int64


In [9]:
# ============================================================
# VALIDATE CUSTOMER STATUS FLAGS
# ============================================================

status_check = pd.crosstab(
    df_features["churn"],
    [
        df_features["churn_flag"],
        df_features["active_flag"]
    ]
)

status_check

churn_flag,0,1
active_flag,1,0
churn,,
no,5174,0
yes,0,1869


# CUSTOMER SEGMENT FEATURES

In [10]:
# ============================================================
# CREATE SENIOR CITIZEN LABEL
# ============================================================

df_features["seniorcitizen_label"] = np.where(
    df_features["seniorcitizen"] == 1,
    "Senior Citizen",
    "Non-Senior"
)

print("Senior citizen label created.")

print(
    df_features["seniorcitizen_label"]
    .value_counts()
)

Senior citizen label created.
seniorcitizen_label
Non-Senior        5901
Senior Citizen    1142
Name: count, dtype: int64


In [13]:
# ============================================================
# CREATE TENURE GROUP
# ============================================================

df_features["tenure_group"] = pd.cut(
    df_features["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-72 Months"
    ]
)

print("Tenure groups created.")

print(
    df_features["tenure_group"]
    .value_counts()
    .sort_index()
)

Tenure groups created.
tenure_group
0-12 Months     2186
13-24 Months    1024
25-48 Months    1594
49-72 Months    2239
Name: count, dtype: int64


In [14]:
# ============================================================
# CREATE MONTHLY CHARGES GROUP
# ============================================================

df_features["monthly_charges_group"] = pd.cut(
    df_features["monthlycharges"],
    bins=[0, 30, 60, 90, 120],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ],
    include_lowest=True
)

print("Monthly charges groups created.")

print(
    df_features["monthly_charges_group"]
    .value_counts()
    .sort_index()
)

Monthly charges groups created.
monthly_charges_group
Low          1653
Medium       1265
High         2386
Very High    1739
Name: count, dtype: int64


In [15]:
# ============================================================
# CREATE TOTAL CHARGES GROUP
# ============================================================

df_features["total_charges_group"] = pd.cut(
    df_features["totalcharges"],
    bins=[-1, 1000, 3000, 6000, np.inf],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

print("Total charges groups created.")

print(
    df_features["total_charges_group"]
    .value_counts()
    .sort_index()
)

Total charges groups created.
total_charges_group
Low          2904
Medium       1935
High         1512
Very High     692
Name: count, dtype: int64


# SERVICE FEATURES

In [16]:
# ============================================================
# CREATE INTERNET SERVICE FLAG
# ============================================================

df_features["has_internet"] = (
    df_features["internetservice"]
    .astype(str)
    .str.lower()
    .ne("no")
    .astype(int)
)

print("Internet service flag created.")

print(
    df_features["has_internet"]
    .value_counts()
    .sort_index()
)

Internet service flag created.
has_internet
0    1526
1    5517
Name: count, dtype: int64


In [17]:
# ============================================================
# CREATE SERVICE COUNT
# ============================================================

service_columns = [
    "phoneservice",
    "onlinesecurity",
    "onlinebackup",
    "deviceprotection",
    "techsupport",
    "streamingtv",
    "streamingmovies"
]

df_features["service_count"] = 0

for column in service_columns:
    df_features["service_count"] += (
        df_features[column]
        .astype(str)
        .str.lower()
        .eq("yes")
        .astype(int)
    )

print("Service count created.")

print(
    df_features["service_count"]
    .value_counts()
    .sort_index()
)

Service count created.
service_count
0      80
1    2253
2     996
3    1041
4    1062
5     827
6     525
7     259
Name: count, dtype: int64


In [18]:
# ============================================================
# CREATE STREAMING SERVICE FLAG
# ============================================================

streaming_columns = [
    "streamingtv",
    "streamingmovies"
]

df_features["has_streaming"] = (
    df_features[streaming_columns]
    .apply(
        lambda row: any(
            str(value).lower() == "yes"
            for value in row
        ),
        axis=1
    )
    .astype(int)
)

print("Streaming service flag created.")

print(
    df_features["has_streaming"]
    .value_counts()
    .sort_index()
)

Streaming service flag created.
has_streaming
0    3544
1    3499
Name: count, dtype: int64


In [19]:
# ============================================================
# CREATE SECURITY & SUPPORT COUNT
# ============================================================

security_support_columns = [
    "onlinesecurity",
    "onlinebackup",
    "deviceprotection",
    "techsupport"
]

df_features["security_support_count"] = 0

for column in security_support_columns:
    df_features["security_support_count"] += (
        df_features[column]
        .astype(str)
        .str.lower()
        .eq("yes")
        .astype(int)
    )

print("Security & support count created.")

print(
    df_features["security_support_count"]
    .value_counts()
    .sort_index()
)

Security & support count created.
security_support_count
0    2793
1    1467
2    1372
3     941
4     470
Name: count, dtype: int64


# PAYMENT FEATURE

In [20]:
# ============================================================
# CREATE PAYMENT METHOD GROUP
# ============================================================

def payment_group(payment):
    payment = str(payment).strip().lower()

    if payment == "electronic check":
        return "Electronic Check"

    elif "automatic" in payment:
        return "Automatic Payment"

    elif payment == "mailed check":
        return "Mailed Check"

    else:
        return "Other"


df_features["payment_group"] = (
    df_features["paymentmethod"]
    .apply(payment_group)
)

print("Payment method groups created.")

print(
    df_features["payment_group"]
    .value_counts()
)

Payment method groups created.
payment_group
Automatic Payment    3066
Electronic Check     2365
Mailed Check         1612
Name: count, dtype: int64


# REVENUE FEATURES

In [21]:
# ============================================================
# CREATE CUSTOMER LIFETIME VALUE
# ============================================================

df_features["customer_lifetime_value"] = (
    df_features["totalcharges"]
)

print("Customer lifetime value created.")

print(
    df_features["customer_lifetime_value"]
    .describe()
    .round(2)
)

Customer lifetime value created.
count    7043.00
mean     2279.73
std      2266.79
min         0.00
25%       398.55
50%      1394.55
75%      3786.60
max      8684.80
Name: customer_lifetime_value, dtype: float64


In [22]:
# ============================================================
# CREATE REVENUE PER TENURE
# ============================================================

df_features["revenue_per_tenure"] = np.where(
    df_features["tenure"] > 0,
    df_features["totalcharges"] / df_features["tenure"],
    df_features["monthlycharges"]
)

print("Revenue per tenure created.")

print(
    df_features["revenue_per_tenure"]
    .describe()
    .round(2)
)

Revenue per tenure created.
count    7043.00
mean       64.76
std        30.19
min        13.78
25%        35.94
50%        70.34
75%        90.17
max       121.40
Name: revenue_per_tenure, dtype: float64


# RISK FEATURES

In [23]:
# ============================================================
# CREATE MONTHLY REVENUE AT RISK
# ============================================================

df_features["monthly_revenue_at_risk"] = np.where(
    df_features["active_flag"] == 1,
    df_features["monthlycharges"],
    0
)

print("Monthly revenue at risk created.")

print(
    f"Monthly Revenue at Risk: "
    f"€{df_features['monthly_revenue_at_risk'].sum():,.2f}"
)

Monthly revenue at risk created.
Monthly Revenue at Risk: €316,985.75


In [24]:
# ============================================================
# CREATE CUSTOMER RISK SCORE
# ============================================================

df_features["risk_score"] = (
    (
        df_features["contract"]
        .astype(str)
        .str.lower()
        .eq("month-to-month")
    ).astype(int) * 30

    +

    (
        df_features["tenure"] <= 12
    ).astype(int) * 20

    +

    (
        df_features["monthlycharges"] >= 80
    ).astype(int) * 20

    +

    (
        df_features["paymentmethod"]
        .astype(str)
        .str.lower()
        .eq("electronic check")
    ).astype(int) * 20

    +

    (
        df_features["security_support_count"] == 0
    ).astype(int) * 10
)

print("Customer risk score created.")

print(
    df_features["risk_score"]
    .describe()
    .round(2)
)

Customer risk score created.
count    7043.00
mean       41.00
std        27.72
min         0.00
25%        20.00
50%        40.00
75%        60.00
max       100.00
Name: risk_score, dtype: float64


In [25]:
# ============================================================
# CREATE RISK CATEGORY
# ============================================================

df_features["risk_category"] = pd.cut(
    df_features["risk_score"],
    bins=[-1, 29, 59, 79, 100],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk",
        "Critical Risk"
    ]
)

print("Risk categories created.")

print(
    df_features["risk_category"]
    .value_counts()
    .sort_index()
)

Risk categories created.
risk_category
Low Risk         2637
Medium Risk      1942
High Risk        1567
Critical Risk     897
Name: count, dtype: int64


# FEATURE VALIDATION

In [26]:
# ============================================================
# CELL 23 — VALIDATE BINARY FEATURES
# ============================================================

binary_features = [
    "churn_flag",
    "active_flag",
    "has_internet",
    "has_streaming"
]

binary_validation = pd.DataFrame({
    "Min": df_features[binary_features].min(),
    "Max": df_features[binary_features].max(),
    "Missing": df_features[binary_features].isna().sum(),
    "Unique": df_features[binary_features].nunique()
})

binary_validation

,Min,Max,Missing,Unique
churn_flag,0,1,0,2
active_flag,0,1,0,2
has_internet,0,1,0,2
has_streaming,0,1,0,2


In [27]:
# ============================================================
# VALIDATE RISK FEATURES
# ============================================================

risk_validation = pd.DataFrame({
    "Min": df_features[
        [
            "risk_score",
            "monthly_revenue_at_risk"
        ]
    ].min(),

    "Max": df_features[
        [
            "risk_score",
            "monthly_revenue_at_risk"
        ]
    ].max(),

    "Missing": df_features[
        [
            "risk_score",
            "monthly_revenue_at_risk"
        ]
    ].isna().sum()
})

risk_validation

,Min,Max,Missing
risk_score,0.0,100.00,0
monthly_revenue_at_risk,0.0,118.75,0


In [28]:
# ============================================================
# FEATURE MISSING VALUE CHECK
# ============================================================

feature_missing = df_features.isna().sum()

feature_missing = feature_missing[
    feature_missing > 0
]

print("Feature missing value check")
print("=" * 60)

if feature_missing.empty:
    print("No missing values found.")
else:
    print(feature_missing)

Feature missing value check
No missing values found.


In [29]:
# ============================================================
# FEATURE DATA TYPES
# ============================================================

feature_columns = [
    "churn_flag",
    "active_flag",
    "seniorcitizen_label",
    "tenure_group",
    "monthly_charges_group",
    "total_charges_group",
    "has_internet",
    "service_count",
    "has_streaming",
    "security_support_count",
    "payment_group",
    "customer_lifetime_value",
    "revenue_per_tenure",
    "monthly_revenue_at_risk",
    "risk_score",
    "risk_category"
]

feature_dtype_summary = pd.DataFrame({
    "Data_Type": df_features[feature_columns]
    .dtypes
    .astype(str),

    "Missing": df_features[feature_columns]
    .isna()
    .sum(),

    "Unique": df_features[feature_columns]
    .nunique()
})

feature_dtype_summary

,Data_Type,Missing,Unique
churn_flag,int64,0,2
active_flag,int64,0,2
seniorcitizen_label,str,0,2
tenure_group,category,0,4
monthly_charges_group,category,0,4
total_charges_group,category,0,4
has_internet,int64,0,2
service_count,int64,0,8
has_streaming,int64,0,2
security_support_count,int64,0,5


# FINAL OVERVIEW

In [30]:
# ============================================================
# FINAL FEATURE LIST
# ============================================================

new_features = [
    column
    for column in df_features.columns
    if column not in df.columns
]

print("=" * 70)
print("NEW ENGINEERED FEATURES")
print("=" * 70)

for feature in new_features:
    print(f"- {feature}")

print("=" * 70)

print(f"Original Columns : {len(df.columns)}")
print(f"New Features     : {len(new_features)}")
print(f"Final Columns    : {len(df_features.columns)}")

NEW ENGINEERED FEATURES
- churn_flag
- active_flag
- seniorcitizen_label
- tenure_group
- monthly_charges_group
- total_charges_group
- has_internet
- service_count
- has_streaming
- security_support_count
- payment_group
- customer_lifetime_value
- revenue_per_tenure
- monthly_revenue_at_risk
- risk_score
- risk_category
Original Columns : 21
New Features     : 16
Final Columns    : 37


In [31]:
# ============================================================
# FINAL FEATURE STATISTICS
# ============================================================

numeric_features = [
    "churn_flag",
    "active_flag",
    "seniorcitizen",
    "tenure",
    "monthlycharges",
    "totalcharges",
    "has_internet",
    "service_count",
    "has_streaming",
    "security_support_count",
    "customer_lifetime_value",
    "revenue_per_tenure",
    "monthly_revenue_at_risk",
    "risk_score"
]

df_features[numeric_features].describe().round(2)

,churn_flag,active_flag,seniorcitizen,tenure,monthlycharges,totalcharges,has_internet,service_count,has_streaming,security_support_count,customer_lifetime_value,revenue_per_tenure,monthly_revenue_at_risk,risk_score
count,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.0,7043.00,7043.00,7043.00,7043.00,7043.00
mean,0.27,0.73,0.16,32.37,64.76,2279.73,0.78,2.94,0.5,1.27,2279.73,64.76,45.01,41.00
std,0.44,0.44,0.37,24.56,30.09,2266.79,0.41,1.84,0.5,1.29,2266.79,30.19,37.97,27.72
min,0.00,0.00,0.00,0.00,18.25,0.00,0.00,0.00,0.0,0.00,0.00,13.78,0.00,0.00
25%,0.00,0.00,0.00,9.00,35.50,398.55,1.00,1.00,0.0,0.00,398.55,35.94,0.00,20.00
50%,0.00,1.00,0.00,29.00,70.35,1394.55,1.00,3.00,0.0,1.00,1394.55,70.34,40.75,40.00
75%,1.00,1.00,0.00,55.00,89.85,3786.60,1.00,4.00,1.0,2.00,3786.60,90.17,80.00,60.00
max,1.00,1.00,1.00,72.00,118.75,8684.80,1.00,7.00,1.0,4.00,8684.80,121.40,118.75,100.00


In [32]:
# ============================================================
# FINAL QUALITY CHECK
# ============================================================

print("=" * 70)
print("FINAL FEATURE ENGINEERING QUALITY CHECK")
print("=" * 70)

print(f"Rows              : {len(df_features):,}")
print(f"Columns           : {len(df_features.columns):,}")
print(f"Missing Values    : {df_features.isna().sum().sum():,}")
print(f"Duplicate Rows    : {df_features.duplicated().sum():,}")
print(
    f"Unique Customers  : "
    f"{df_features['customerid'].nunique():,}"
)

print("=" * 70)

FINAL FEATURE ENGINEERING QUALITY CHECK
Rows              : 7,043
Columns           : 37
Missing Values    : 0
Duplicate Rows    : 0
Unique Customers  : 7,043


In [33]:
# ============================================================
# SAVE FEATURE DATASET
# ============================================================

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    OUTPUT_DIR /
    "customer_churn_features.csv"
)

df_features.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Feature dataset saved successfully.")
print(f"Path: {OUTPUT_PATH}")
print(f"Rows: {df_features.shape[0]:,}")
print(f"Columns: {df_features.shape[1]:,}")

Feature dataset saved successfully.
Path: ..\data\processed\customer_churn_features.csv
Rows: 7,043
Columns: 37


In [34]:
# ============================================================
# RELOAD VALIDATION
# ============================================================

df_check = pd.read_csv(OUTPUT_PATH)

print("=" * 70)
print("RELOAD VALIDATION")
print("=" * 70)

print(f"Rows             : {df_check.shape[0]:,}")
print(f"Columns          : {df_check.shape[1]:,}")
print(f"Missing Values   : {df_check.isna().sum().sum():,}")
print(f"Duplicate Rows   : {df_check.duplicated().sum():,}")
print(
    f"Unique Customers : "
    f"{df_check['customerid'].nunique():,}"
)

print("=" * 70)

RELOAD VALIDATION
Rows             : 7,043
Columns          : 37
Missing Values   : 0
Duplicate Rows   : 0
Unique Customers : 7,043
